In [ ]:
"""
Control de Instrumentos de Medición con VISA
============================================

Este script establece la comunicación entre la PC y un instrumento de medición,
usando PyVISA y realiza operaciones básicas de configuración.

Funcionalidades principales:
- Detección automática de instrumentos conectados.
- Identificación del instrumento.
- Consulta y modificación de parámetros de configuración.

Autor: Tardón, Damián David
Fecha: 06/09/2025
Versión: 1.0
"""

# ============================================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================================
# Estas librerías son necesarias para la comunicación VISA:
# - pyvisa: Interface principal para comunicación con instrumentos.
# - pyvisa-py: Backend puro de Python (no requiere VISA runtime).
# - pyserial: Para comunicación serial (usado por pyvisa-py).

!pip install pyvisa pyvisa-py pyserial

In [ ]:
# ============================================================================
# IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN INICIAL
# ============================================================================

import pyvisa

# Crear el administrador de recursos VISA.
# Este objeto gestiona todas las conexiones a instrumentos.

rm = pyvisa.ResourceManager()

In [ ]:
# ============================================================================
# DETECCIÓN DE INSTRUMENTOS CONECTADOS
# ============================================================================

print("=== DETECCIÓN DE INSTRUMENTOS ===")

# Escanear todos los recursos VISA disponibles en el sistema.
# Incluye puertos serie, USB, Ethernet, etc.

instrumentos = rm.list_resources()
print("Instrumentos encontrados:", instrumentos)

In [ ]:
# ============================================================================
# IDENTIFICACIÓN DEL INSTRUMENTO
# ============================================================================
# Nomenclatura VISA:
# ASRL4::INSTR = Puerto serie COM4 (ASRL = Asynchronous Serial).
# USB0::INSTR = Dispositivo USB.
# TCPIP::ip_address::INSTR = Dispositivo de red.

print("\n=== IDENTIFICACIÓN DEL INSTRUMENTO ===")

try:
    # Abrir conexión con el instrumento seleccionado.
    # Establece un canal de comunicación bidireccional.
    # ASRL4::INSTR corresponde al puerto COM4 en Windows.
    instrumento = rm.open_resource('ASRL4::INSTR')
    
    # *IDN? es un comando estándar SCPI.
    # Solicita identificación (fabricante, modelo, etc.).
    # SCPI (Standard Commands for Programmable Instruments) es un protocolo estándar, basado en IEEE 488.2.
    # query() envía el comando y espera la respuesta del instrumento.
    ID = instrumento.query('*IDN?')
    print("ID del instrumento:", ID)
    
    # Cerrar la conexión para liberar recursos.
    instrumento.close()
    
except Exception as e:
    print("Error al conectar con el instrumento:", e)

In [ ]:
# ============================================================================
# CONSULTA DE CONFIGURACIÓN ACTUAL
# ============================================================================
print("\n=== CONSULTA DE PARÁMETROS ===")

try:
    # Abrir conexión con el instrumento seleccionado.
    instrumento = rm.open_resource('ASRL4::INSTR')
    
    # :TIMebase:SCALe? es un comando estándar SCPI.
    # Solicita el valor actual de la escala de tiempo (tiempo por división) del osciloscopio.
    escala_tiempo = instrumento.query(':TIMebase:SCALe?')
    print(f"Escala de tiempo actual: {escala_tiempo}")
    
    # Cerrar la conexión para liberar recursos.
    instrumento.close()
    
except Exception as e:
    print("Error al consultar configuración:", e)

In [ ]:
# ============================================================================
# MODIFICACIÓN DE CONFIGURACIÓN
# ============================================================================
print("\n=== MODIFICACIÓN DE CONFIGURACIÓN ===")

try:
    # Abrir conexión con el instrumento seleccionado.
    instrumento = rm.open_resource('ASRL4::INSTR')
    
    nuevo_valor = 5E-3  # 5 ms por división.
    print(f"Cambiando escala de tiempo a {nuevo_valor}s/div...")

    # :TIMebase:SCALe 5E-3 es un comando estándar SCPI.
    # Establece la escala de tiempo.
    # write() envía el comando sin esperar respuesta.
    instrumento.write(':TIMebase:SCALe 5E-3')  # 5ms por división.
    # instrumento.write(f':TIMebase:SCALe {nuevo_valor}')  # 5ms por división.

    # Verificar que el cambio se aplicó correctamente.
    escala_tiempo = instrumento.query(':TIMebase:SCALe?')
    if float(escala_tiempo) == nuevo_valor:
        print("✓ La configuración se aplicó correctamente.")
    else:
        print("⚠ La configuración no se aplicó correctamente.")
    print(f"Escala de tiempo: {escala_tiempo} s/div")

    # Cerrar la conexión para liberar recursos.
    instrumento.close()
    
except Exception as e:
    print("Error al modificar configuración:", e)
    print("Verificar que el comando sea correcto.")

In [ ]:
# ============================================================================
# LIMPIEZA FINAL
# ============================================================================
# Cerrar el administrador de recursos y liberar memoria.
rm.close()
print("\n=== OPERACIÓN COMPLETADA ===")
print("Conexiones cerradas correctamente")

In [ ]:
# ============================================================================
# LISTA DE COMANDOS SCPI COMUNES
# ============================================================================
"""
'*IDN?'
descripción: Solicita identificación del instrumento
respuesta_típica: GW, GDS-1152A-U,N°serie, V1.00
uso: Verificar que el instrumento está conectado y respondiendo

':TIMebase:SCALe?'
descripción: Consulta la escala de tiempo actual (segundos por división).
respuesta_típica: 1.00E-03 (1 milisegundo por división).
uso: Conocer la configuración actual del eje temporal.

':TIMebase:SCALe <valor>'
descripción: Establece la escala de tiempo.
ejemplo: :TIMebase:SCALe 5E-3 (5 milisegundos por división).
uso: Ajustar la resolución temporal de las mediciones.


"""

In [ ]:
# ============================================================================
# MEJORAS SUGERIDAS PARA CÓDIGO DE PRODUCCIÓN
# ============================================================================
"""
Para un código más robusto, considerar:

1. GESTIÓN DE CONTEXTO:
   with rm.open_resource('ASRL4::INSTR') as instrumento:
       # operaciones aquí
       # se cierra automáticamente

2. CONFIGURACIÓN DE TIMEOUT:
   instrumento.timeout = 5000  # 5 segundos

3. VALIDACIÓN DE RESPUESTAS:
   if respuesta.strip():
       # procesar respuesta válida

4. LOGGING:
   import logging
   logging.info("Conectando con instrumento...")

5. CONFIGURACIÓN CENTRALIZADA:
   PUERTO_INSTRUMENTO = 'ASRL4::INSTR'
   TIMEOUT_MS = 5000
"""